# Comment Sentiment, Emotion, Toxicity, Topic, and LLooM Analysis

This notebook combines the useful pieces of the supplied comment-analysis scripts and theme-analysis notebook into one workflow for a CSV of comments.

### What it does
1. Loads and cleans comment text while preserving the original text.
2. Runs **sentiment analysis** (positive / neutral / negative).
3. Runs **emotion analysis** (anger, disgust, fear, joy, neutral, sadness, surprise).
4. Optionally runs **toxicity scoring**.
5. Produces comment-level and aggregate summaries.
6. Runs lexical analysis: word frequencies, n-grams, TF-IDF, NMF themes, and word clouds.
7. Optionally runs **BERTopic** semantic topic discovery.
8. Runs **LLooM concept induction** to surface higher-level, human-readable concepts and compare them with sentiment/emotion metadata.
9. Exports reusable CSV files and plots.

### Start here
For most datasets, you only need to change `INPUT_CSV` in the configuration cell. If auto-detection chooses the wrong text column, set `TEXT_COL` explicitly.

The notebook does **not** remove duplicate comments by default. For public-comment datasets, repeated/template comments can be substantively meaningful because their frequency is part of the observed corpus.

## 0. Optional package installation

Uncomment and run the installation line if your environment does not already have the required packages.

`text_lloom` is only needed for the LLooM section. LLooM uses an LLM API and can incur API cost, so that section is controlled by its own switch.

In [ ]:
# Uncomment if needed:
# %pip install -q pandas numpy matplotlib scikit-learn transformers torch tqdm wordcloud \
#     bertopic sentence-transformers umap-learn hdbscan text_lloom python-dotenv

## 1. Configuration

In [ ]:
from pathlib import Path

# -----------------------------
# Required
# -----------------------------
INPUT_CSV = Path("comments.csv")   # <-- CHANGE THIS

# Set to None to auto-detect.
TEXT_COL = None
ID_COL = None

# Optional grouping column for aggregate analysis.
# Example: "video_id", "docketId", "category", etc.
GROUP_COL = None

OUTPUT_DIR = Path("comment_analysis_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Cleaning
# -----------------------------
DROP_EMPTY_COMMENTS = True
DROP_EXACT_DUPLICATES = False

# -----------------------------
# Transformer analyses
# -----------------------------
RUN_SENTIMENT = True
RUN_EMOTION = True
RUN_TOXICITY = True

SENTIMENT_MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"
EMOTION_MODEL = "j-hartmann/emotion-english-distilroberta-base"
TOXICITY_MODEL = "unitary/toxic-bert"

BATCH_SIZE = 32
MAX_LENGTH = 512

# Used only when creating GROUP_COL summaries.
# 1.0 keeps every neutral comment, matching the supplied analytics script's
# current invocation. Change to 0.50 if you want to retain only half of neutral
# comments when computing group-level summaries.
NEUTRAL_KEEP_FRAC = 1.0
RANDOM_SEED = 42

# -----------------------------
# Topic/theme analyses
# -----------------------------
N_NMF_THEMES = 8
RUN_BERTOPIC = True
N_BERTOPIC_CATEGORIES = 10

# -----------------------------
# LLooM
# -----------------------------
RUN_LLOOM = False  # Set True only after OPENAI_API_KEY is configured.
LLOOM_DISCOVERY_SAMPLE = 1000
LLOOM_MAX_CONCEPTS = 8
LLOOM_SEED = "arguments, concerns, support, opposition, and perceived impacts expressed in the comments"

# LLooM scoring across a very large corpus can be expensive.
# False = generate/score concepts on the discovery sample only.
# True = after concept induction, score the entire cleaned dataframe.
LLOOM_SCORE_FULL_DATASET = False
LLOOM_SCORE_BATCH_SIZE = 5

## 2. Imports and environment check

In [ ]:
import os
import re
import html
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 160)
warnings.filterwarnings("ignore", category=FutureWarning)

try:
    import torch
    DEVICE = 0 if torch.cuda.is_available() else -1
    DEVICE_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
except Exception:
    DEVICE = -1
    DEVICE_NAME = "CPU"

print("Inference device:", DEVICE_NAME)
print("Output directory:", OUTPUT_DIR.resolve())

## 3. Load comments and detect the text column

The loader recognizes several common names, including `comment_text`, `COMMENT TEXT`, `comment`, `text`, and flattened Regulations.gov-style names.

In [ ]:
if not INPUT_CSV.exists():
    raise FileNotFoundError(
        f"Could not find {INPUT_CSV}. Update INPUT_CSV in the configuration cell."
    )

raw_df = pd.read_csv(INPUT_CSV, low_memory=False)
print(f"Loaded {len(raw_df):,} rows x {raw_df.shape[1]:,} columns")
print("\nColumns:")
print(raw_df.columns.tolist())

TEXT_CANDIDATES = [
    "comment_text", "COMMENT TEXT", "Comment Text", "comment", "Comment",
    "comments", "Comments", "text", "Text", "attributes.comment",
    "attributes_comment", "commentText", "comment_body", "body",
]

ID_CANDIDATES = [
    "comment_id", "commentId", "document_id", "documentId",
    "id", "ID", "objectId",
]

def first_existing(candidates, columns):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None

if TEXT_COL is None:
    TEXT_COL = first_existing(TEXT_CANDIDATES, raw_df.columns)

if TEXT_COL is None:
    raise ValueError(
        "Could not auto-detect the comment text column. "
        "Set TEXT_COL explicitly in the configuration cell."
    )

if ID_COL is None:
    ID_COL = first_existing(ID_CANDIDATES, raw_df.columns)

df = raw_df.copy()

if ID_COL is None:
    ID_COL = "comment_row_id"
    df[ID_COL] = np.arange(1, len(df) + 1)

if GROUP_COL is None and "video_id" in df.columns:
    GROUP_COL = "video_id"

print(f"\nUsing TEXT_COL = {TEXT_COL!r}")
print(f"Using ID_COL   = {ID_COL!r}")
print(f"Using GROUP_COL = {GROUP_COL!r}")

## 4. Clean comment text

Two text fields are retained:

- `comment_text_clean`: minimally cleaned text used for sentiment, emotion, toxicity, BERTopic, and LLooM.
- `comment_text_nlp`: more aggressively normalized text used for word counts, TF-IDF, NMF, and n-grams.

HTML entities such as `&ldquo;` are decoded before analysis.

In [ ]:
def clean_comment_text(value):
    if pd.isna(value):
        return ""
    text = html.unescape(str(value))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

DEFAULT_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "being", "by",
    "for", "from", "had", "has", "have", "he", "her", "hers", "him", "his",
    "i", "if", "in", "into", "is", "it", "its", "me", "my", "of", "on",
    "or", "our", "ours", "she", "that", "the", "their", "theirs", "them",
    "they", "this", "to", "was", "we", "were", "what", "when", "where",
    "which", "who", "will", "with", "would", "you", "your", "yours",
    "http", "https", "www", "url", "amp"
}
EXTRA_STOPWORDS = set()
STOPWORDS = DEFAULT_STOPWORDS | EXTRA_STOPWORDS

def normalize_for_topics(text):
    text = clean_comment_text(text).lower()
    text = text.replace("_", " ").replace("/", " ")
    text = re.sub(r"[^a-z0-9'#+.\- ]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def tokenize_for_topics(text, min_len=3):
    raw_terms = re.findall(r"[a-z][a-z0-9'#+.\-]*", normalize_for_topics(text))
    terms = [t.strip("'#+.-") for t in raw_terms]
    return [
        t for t in terms
        if len(t) >= min_len
        and t not in STOPWORDS
        and not t.isdigit()
    ]

def clean_for_topics(text):
    return " ".join(tokenize_for_topics(text))

df["comment_text_raw"] = df[TEXT_COL]
df["comment_text_clean"] = df[TEXT_COL].apply(clean_comment_text)
df["comment_text_nlp"] = df["comment_text_clean"].apply(clean_for_topics)
df["comment_word_count"] = df["comment_text_clean"].str.findall(r"\b\w+\b").str.len()
df["is_exact_duplicate_text"] = df.duplicated("comment_text_clean", keep=False)

if DROP_EMPTY_COMMENTS:
    df = df[df["comment_text_clean"].ne("")].copy()

if DROP_EXACT_DUPLICATES:
    df = df.drop_duplicates("comment_text_clean", keep="first").copy()

df = df.reset_index(drop=True)

print(f"Rows retained: {len(df):,}")
print(f"Exact-duplicate rows present: {int(df['is_exact_duplicate_text'].sum()):,}")
display(df[[ID_COL, "comment_text_clean", "comment_word_count"]].head())

## 5. Basic corpus diagnostics

In [ ]:
corpus_summary = pd.DataFrame({
    "metric": [
        "comments", "nonempty_comments", "exact_duplicate_rows",
        "median_words", "mean_words", "p90_words",
    ],
    "value": [
        len(df),
        int(df["comment_text_clean"].ne("").sum()),
        int(df["is_exact_duplicate_text"].sum()),
        float(df["comment_word_count"].median()),
        float(df["comment_word_count"].mean()),
        float(df["comment_word_count"].quantile(0.90)),
    ],
})

display(corpus_summary)
corpus_summary.to_csv(OUTPUT_DIR / "corpus_summary.csv", index=False)

# Part A — Sentiment, emotion, and toxicity

The supplied emotion script uses `j-hartmann/emotion-english-distilroberta-base`; this notebook keeps that model and adds sentiment probabilities and a signed sentiment value.

`sentiment_signed` is:
- `-1` = negative
- `0` = neutral
- `+1` = positive

`sentiment_continuous = P(positive) - P(negative)` gives a smoother score from `-1` to `+1`.

## 6. Helper functions for batched transformer inference

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

def ensure_candidate_list(item):
    return [item] if isinstance(item, dict) else item

def run_text_pipeline_in_chunks(
    clf,
    texts,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    chunk_size=2000,
):
    outputs = []
    for start in tqdm(range(0, len(texts), chunk_size), desc="Model chunks"):
        chunk = texts[start:start + chunk_size]
        out = clf(
            chunk,
            batch_size=batch_size,
            truncation=True,
            max_length=max_length,
        )
        outputs.extend(out)
    return outputs

def safe_label(label):
    return re.sub(r"[^a-z0-9]+", "_", str(label).strip().lower()).strip("_")

## 7. Sentiment analysis

In [ ]:
if RUN_SENTIMENT:
    print("Loading sentiment model:", SENTIMENT_MODEL)

    sentiment_pipe = pipeline(
        "text-classification",
        model=SENTIMENT_MODEL,
        tokenizer=SENTIMENT_MODEL,
        device=DEVICE,
        top_k=None,
    )

    sentiment_outputs = run_text_pipeline_in_chunks(
        sentiment_pipe,
        df["comment_text_clean"].tolist(),
    )

    fallback_map = {
        "label_0": "negative",
        "label_1": "neutral",
        "label_2": "positive",
    }

    top_labels, top_scores, signed, continuous = [], [], [], []

    for row_i, candidates in enumerate(sentiment_outputs):
        candidates = ensure_candidate_list(candidates)
        score_map = {}

        for candidate in candidates:
            raw_label = safe_label(candidate["label"])
            label = fallback_map.get(raw_label, raw_label)
            score = float(candidate["score"])
            score_map[label] = score
            df.at[row_i, f"sentiment_{label}"] = score

        best_label = max(score_map, key=score_map.get)
        top_labels.append(best_label)
        top_scores.append(score_map[best_label])
        signed.append({"negative": -1, "neutral": 0, "positive": 1}.get(best_label, np.nan))
        continuous.append(score_map.get("positive", 0.0) - score_map.get("negative", 0.0))

    df["sentiment_label"] = top_labels
    df["sentiment_confidence"] = top_scores
    df["sentiment_signed"] = signed
    df["sentiment_continuous"] = continuous

    sentiment_summary = (
        df.groupby("sentiment_label", dropna=False)
          .agg(
              comments=("comment_text_clean", "size"),
              mean_confidence=("sentiment_confidence", "mean"),
              mean_continuous=("sentiment_continuous", "mean"),
          )
          .reset_index()
          .sort_values("comments", ascending=False)
    )
    sentiment_summary["fraction"] = sentiment_summary["comments"] / len(df)

    display(sentiment_summary)
    sentiment_summary.to_csv(OUTPUT_DIR / "sentiment_summary.csv", index=False)
else:
    print("RUN_SENTIMENT is False — skipped.")

## 8. Emotion analysis

In [ ]:
if RUN_EMOTION:
    print("Loading emotion model:", EMOTION_MODEL)

    emotion_pipe = pipeline(
        "text-classification",
        model=EMOTION_MODEL,
        tokenizer=EMOTION_MODEL,
        device=DEVICE,
        top_k=None,
    )

    emotion_outputs = run_text_pipeline_in_chunks(
        emotion_pipe,
        df["comment_text_clean"].tolist(),
    )

    top_labels, top_scores = [], []

    for row_i, candidates in enumerate(emotion_outputs):
        candidates = ensure_candidate_list(candidates)
        best = max(candidates, key=lambda x: float(x["score"]))
        top_labels.append(safe_label(best["label"]))
        top_scores.append(float(best["score"]))

        for candidate in candidates:
            label = safe_label(candidate["label"])
            df.at[row_i, f"emotion_{label}"] = float(candidate["score"])

    df["emotion_label"] = top_labels
    df["emotion_confidence"] = top_scores

    emotion_summary = (
        df.groupby("emotion_label", dropna=False)
          .agg(
              comments=("comment_text_clean", "size"),
              mean_confidence=("emotion_confidence", "mean"),
          )
          .reset_index()
          .sort_values("comments", ascending=False)
    )
    emotion_summary["fraction"] = emotion_summary["comments"] / len(df)

    display(emotion_summary)
    emotion_summary.to_csv(OUTPUT_DIR / "emotion_summary.csv", index=False)
else:
    print("RUN_EMOTION is False — skipped.")

## 9. Toxicity analysis

This keeps the toxicity dimension used by the supplied aggregation script. `unitary/toxic-bert` is a multi-label model, so this cell applies a sigmoid to each label rather than treating the labels as mutually exclusive.

In [ ]:
if RUN_TOXICITY:
    if "torch" not in globals():
        raise ImportError("PyTorch is required for toxicity scoring.")

    print("Loading toxicity model:", TOXICITY_MODEL)

    tox_tokenizer = AutoTokenizer.from_pretrained(TOXICITY_MODEL)
    tox_model = AutoModelForSequenceClassification.from_pretrained(TOXICITY_MODEL)

    torch_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tox_model.to(torch_device)
    tox_model.eval()

    id2label = {int(k): safe_label(v) for k, v in tox_model.config.id2label.items()}

    all_prob_rows = []
    texts = df["comment_text_clean"].tolist()

    with torch.no_grad():
        for start in tqdm(range(0, len(texts), BATCH_SIZE), desc="Toxicity batches"):
            batch_texts = texts[start:start + BATCH_SIZE]
            encoded = tox_tokenizer(
                batch_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
            )
            encoded = {k: v.to(torch_device) for k, v in encoded.items()}
            logits = tox_model(**encoded).logits
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            all_prob_rows.append(probs)

    tox_probs = np.vstack(all_prob_rows)

    for label_id, label_name in id2label.items():
        df[f"toxicity_{label_name}"] = tox_probs[:, label_id]

    toxic_ids = [i for i, label in id2label.items() if label == "toxic"]
    toxic_idx = toxic_ids[0] if toxic_ids else 0

    if not toxic_ids:
        print("Warning: exact 'toxic' label was not found; using the first model output.")

    df["toxicity"] = tox_probs[:, toxic_idx]
    df["toxicity_flag_0_5"] = df["toxicity"] >= 0.5

    toxicity_summary = pd.DataFrame({
        "metric": ["mean", "median", "p90", "p95", "max", "fraction_at_or_above_0.5"],
        "value": [
            df["toxicity"].mean(),
            df["toxicity"].median(),
            df["toxicity"].quantile(0.90),
            df["toxicity"].quantile(0.95),
            df["toxicity"].max(),
            df["toxicity_flag_0_5"].mean(),
        ],
    })

    display(toxicity_summary)
    toxicity_summary.to_csv(OUTPUT_DIR / "toxicity_summary.csv", index=False)
else:
    print("RUN_TOXICITY is False — skipped.")

## 10. Sentiment / emotion / toxicity plots

In [ ]:
PLOT_DIR = OUTPUT_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

def save_current_plot(filename):
    plt.tight_layout()
    plt.savefig(PLOT_DIR / filename, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()

if "sentiment_label" in df.columns:
    counts = df["sentiment_label"].value_counts().reindex(
        ["negative", "neutral", "positive"]
    ).dropna()

    plt.figure(figsize=(7, 4))
    counts.plot(kind="bar")
    plt.title("Comment Sentiment Distribution")
    plt.xlabel("Sentiment")
    plt.ylabel("Comments")
    plt.xticks(rotation=0)
    save_current_plot("sentiment_distribution.png")

if "emotion_label" in df.columns:
    counts = df["emotion_label"].value_counts()

    plt.figure(figsize=(8, 4.5))
    counts.plot(kind="bar")
    plt.title("Comment Emotion Distribution")
    plt.xlabel("Emotion")
    plt.ylabel("Comments")
    plt.xticks(rotation=35, ha="right")
    save_current_plot("emotion_distribution.png")

if "toxicity" in df.columns:
    plt.figure(figsize=(7, 4))
    plt.hist(df["toxicity"].dropna(), bins=40)
    plt.title("Comment Toxicity Score Distribution")
    plt.xlabel("Toxicity score")
    plt.ylabel("Comments")
    save_current_plot("toxicity_distribution.png")

if {"sentiment_continuous", "toxicity"}.issubset(df.columns):
    sample_n = min(5000, len(df))
    plot_df = df.sample(sample_n, random_state=RANDOM_SEED) if len(df) > sample_n else df

    plt.figure(figsize=(7, 5))
    plt.scatter(
        plot_df["sentiment_continuous"],
        plot_df["toxicity"],
        alpha=0.25,
        s=12,
    )
    plt.title("Sentiment vs. Toxicity")
    plt.xlabel("Sentiment continuous (-1 negative to +1 positive)")
    plt.ylabel("Toxicity")
    save_current_plot("sentiment_vs_toxicity.png")

## 11. Aggregate summary

This reproduces the main logic of the supplied `comment_analytics` script, but works with any optional grouping column.

If `GROUP_COL` is set, the output includes mean/max toxicity, mean signed sentiment, positive/neutral/negative fractions, and dominant emotion when available. Neutral subsampling is controlled by `NEUTRAL_KEEP_FRAC`.

In [ ]:
def build_group_summary(frame, group_col, neutral_keep_frac=1.0, random_seed=42):
    if group_col is None or group_col not in frame.columns:
        return None

    rng = random.Random(random_seed)
    output_rows = []

    for group_value, group in frame.groupby(group_col, dropna=False):
        working = group.copy()

        if "sentiment_signed" in working.columns and neutral_keep_frac < 1.0:
            neutral = working[working["sentiment_signed"] == 0]
            non_neutral = working[working["sentiment_signed"] != 0]

            keep_n = min(int(round(len(neutral) * neutral_keep_frac)), len(neutral))
            if keep_n:
                sampled_idx = rng.sample(list(neutral.index), keep_n)
                sampled_neutral = neutral.loc[sampled_idx]
            else:
                sampled_neutral = neutral.iloc[0:0]

            working = pd.concat([non_neutral, sampled_neutral], axis=0)

        if len(working) == 0:
            continue

        row = {group_col: group_value, "n_comments_used": len(working)}

        if "toxicity" in working.columns:
            row["toxicity_mean"] = working["toxicity"].mean()
            row["toxicity_max"] = working["toxicity"].max()

        if "sentiment_signed" in working.columns:
            s = working["sentiment_signed"]
            row["sentiment_signed_mean"] = s.mean()
            row["sentiment_pos_frac"] = (s > 0).mean()
            row["sentiment_neu_frac"] = (s == 0).mean()
            row["sentiment_neg_frac"] = (s < 0).mean()

        if "emotion_label" in working.columns and working["emotion_label"].notna().any():
            row["dominant_emotion"] = working["emotion_label"].mode().iloc[0]

        output_rows.append(row)

    return pd.DataFrame(output_rows)

group_summary = build_group_summary(
    df,
    GROUP_COL,
    neutral_keep_frac=NEUTRAL_KEEP_FRAC,
    random_seed=RANDOM_SEED,
)

if group_summary is not None:
    display(group_summary.head(20))
    group_summary.to_csv(OUTPUT_DIR / "group_summary.csv", index=False)
else:
    print("No GROUP_COL selected — group-level aggregation skipped.")

## 12. Save the comment-level analysis before topic modeling

In [ ]:
comment_level_path = OUTPUT_DIR / "comment_level_sentiment_emotion_toxicity.csv"
df.to_csv(comment_level_path, index=False)
print("Saved:", comment_level_path)

# Part B — Lexical and topic/theme analysis

## 13. Top words, bigrams, and trigrams

In [ ]:
row_tokens = [tokenize_for_topics(text) for text in df["comment_text_clean"].fillna("")]

unigram_counts = Counter(t for toks in row_tokens for t in toks)
bigram_counts = Counter()
trigram_counts = Counter()

for toks in row_tokens:
    bigram_counts.update(zip(toks, toks[1:]))
    trigram_counts.update(zip(toks, toks[1:], toks[2:]))

rows = []
for term, count in unigram_counts.most_common(100):
    rows.append({"ngram_type": "unigram", "term": term, "count": count})
for terms, count in bigram_counts.most_common(100):
    rows.append({"ngram_type": "bigram", "term": " ".join(terms), "count": count})
for terms, count in trigram_counts.most_common(100):
    rows.append({"ngram_type": "trigram", "term": " ".join(terms), "count": count})

top_phrases = pd.DataFrame(rows)
top_phrases.to_csv(OUTPUT_DIR / "top_words_and_phrases.csv", index=False)
display(top_phrases.head(30))

## 14. TF-IDF terms

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

topic_docs = df["comment_text_nlp"].fillna("").astype(str)
topic_docs = topic_docs[topic_docs.str.strip().ne("")]

if len(topic_docs) < 2:
    raise ValueError("Not enough non-empty comments for TF-IDF/topic analysis.")

MIN_DF = 1 if len(topic_docs) < 30 else 2 if len(topic_docs) < 100 else 3

tfidf_vectorizer = TfidfVectorizer(
    stop_words=list(STOPWORDS),
    ngram_range=(1, 3),
    min_df=MIN_DF,
    max_df=0.95,
    max_features=5000,
)

tfidf_X = tfidf_vectorizer.fit_transform(topic_docs)
tfidf_terms_arr = tfidf_vectorizer.get_feature_names_out()
mean_scores = np.asarray(tfidf_X.mean(axis=0)).ravel()
doc_counts = np.asarray((tfidf_X > 0).sum(axis=0)).ravel()

tfidf_terms = pd.DataFrame({
    "term": tfidf_terms_arr,
    "mean_tfidf": mean_scores,
    "document_count": doc_counts,
}).sort_values(["mean_tfidf", "document_count"], ascending=[False, False])

tfidf_terms.to_csv(OUTPUT_DIR / "tfidf_top_terms_overall.csv", index=False)
display(tfidf_terms.head(30))

## 15. NMF themes + theme word clouds

This keeps the interpretable NMF-theme workflow from the supplied PR notebook, but applies it to comment text.

In [ ]:
from sklearn.decomposition import NMF
from wordcloud import WordCloud

NMF_DIR = OUTPUT_DIR / "nmf_theme_wordclouds"
NMF_DIR.mkdir(parents=True, exist_ok=True)

nmf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words=list(STOPWORDS),
    max_df=0.95,
    min_df=MIN_DF,
    max_features=8000,
    ngram_range=(1, 2),
)

nmf_X = nmf_vectorizer.fit_transform(topic_docs)
nmf_terms = nmf_vectorizer.get_feature_names_out()

n_themes = min(N_NMF_THEMES, nmf_X.shape[0], nmf_X.shape[1])

nmf_model = NMF(
    n_components=n_themes,
    random_state=RANDOM_SEED,
    init="nndsvda",
    max_iter=1000,
)

doc_theme_matrix = nmf_model.fit_transform(nmf_X)
theme_term_matrix = nmf_model.components_

def top_theme_terms(theme_idx, n=12):
    ids = theme_term_matrix[theme_idx].argsort()[::-1][:n]
    return [nmf_terms[i] for i in ids]

dominant_theme = doc_theme_matrix.argmax(axis=1)
theme_strength = doc_theme_matrix.max(axis=1)

theme_labels = {}
theme_rows = []

for i in range(n_themes):
    words = top_theme_terms(i, 12)
    label = f"Theme {i + 1}: " + " / ".join(words[:3])
    theme_labels[i] = label
    theme_rows.append({
        "theme_id": i + 1,
        "theme_label": label,
        "top_terms": ", ".join(words),
        "n_comments": int((dominant_theme == i).sum()),
    })

nmf_theme_summary = (
    pd.DataFrame(theme_rows)
      .sort_values("n_comments", ascending=False)
      .reset_index(drop=True)
)

display(nmf_theme_summary)
nmf_theme_summary.to_csv(OUTPUT_DIR / "nmf_theme_summary.csv", index=False)

df["nmf_theme_id"] = pd.NA
df["nmf_theme_label"] = pd.NA
df["nmf_theme_strength"] = np.nan

df.loc[topic_docs.index, "nmf_theme_id"] = dominant_theme + 1
df.loc[topic_docs.index, "nmf_theme_label"] = [theme_labels[i] for i in dominant_theme]
df.loc[topic_docs.index, "nmf_theme_strength"] = theme_strength

for i in range(n_themes):
    top_ids = theme_term_matrix[i].argsort()[::-1][:150]
    frequencies = {
        nmf_terms[j]: float(theme_term_matrix[i][j])
        for j in top_ids
        if theme_term_matrix[i][j] > 0
    }
    if not frequencies:
        continue

    wc = WordCloud(
        width=1600,
        height=900,
        background_color="white",
        max_words=150,
        collocations=False,
    ).generate_from_frequencies(frequencies)

    plt.figure(figsize=(9, 5))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(theme_labels[i])
    plt.tight_layout()
    plt.savefig(NMF_DIR / f"theme_{i + 1:02d}.png", dpi=180, bbox_inches="tight")
    plt.show()
    plt.close()

df.to_csv(OUTPUT_DIR / "comments_with_nmf_themes.csv", index=False)

## 16. BERTopic semantic categories

BERTopic is optional and can be slower than NMF. It uses the minimally cleaned comment text so semantic context is preserved.

In [ ]:
if RUN_BERTOPIC:
    from bertopic import BERTopic
    from sentence_transformers import SentenceTransformer
    from sklearn.feature_extraction.text import CountVectorizer
    from umap import UMAP
    from hdbscan import HDBSCAN

    bertopic_docs = df["comment_text_clean"].fillna("").astype(str).str.strip()
    bertopic_docs = bertopic_docs[bertopic_docs.ne("")]

    min_topic_size = max(10, min(50, max(10, len(bertopic_docs) // 100)))

    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
    vectorizer_model = CountVectorizer(
        stop_words=list(STOPWORDS),
        ngram_range=(1, 3),
        min_df=MIN_DF,
    )
    umap_model = UMAP(
        n_neighbors=15,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=RANDOM_SEED,
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=min_topic_size,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True,
    )

    bertopic_model = BERTopic(
        embedding_model=embedding_model,
        vectorizer_model=vectorizer_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        nr_topics=N_BERTOPIC_CATEGORIES,
        calculate_probabilities=False,
        verbose=True,
    )

    topic_ids, _ = bertopic_model.fit_transform(bertopic_docs.tolist())

    topic_info = bertopic_model.get_topic_info()
    topic_info.to_csv(OUTPUT_DIR / "bertopic_topic_info.csv", index=False)

    doc_info = bertopic_model.get_document_info(bertopic_docs.tolist())
    doc_info.insert(0, "_original_index", bertopic_docs.index.to_numpy())

    bertopic_assignments = (
        df.loc[bertopic_docs.index, [ID_COL, "comment_text_clean"]]
          .reset_index()
          .rename(columns={"index": "_original_index"})
          .merge(doc_info, on="_original_index", how="left")
    )

    bertopic_assignments.to_csv(
        OUTPUT_DIR / "comments_with_bertopic_categories.csv",
        index=False,
    )

    display(topic_info.head(20))
else:
    print("RUN_BERTOPIC is False — skipped.")

# Part C — LLooM concept induction

LLooM is different from BERTopic/NMF: it uses LLMs to propose **higher-level concepts with natural-language inclusion criteria**, then scores comments against those concepts.

### Recommended workflow for a large corpus
For tens of thousands of comments, first induce concepts from a representative sample. Then inspect those concepts before deciding whether the additional API cost of scoring the full corpus is worthwhile.

Set `OPENAI_API_KEY` in your environment before running this section. Do not paste a key into a notebook you plan to share.

LLooM documentation: https://stanfordhci.github.io/lloom/about/get-started.html

## 17. Prepare a representative LLooM discovery sample

In [ ]:
lloom_source_cols = [ID_COL, "comment_text_clean"]

for optional_col in ["sentiment_label", "sentiment_continuous", "emotion_label", "toxicity"]:
    if optional_col in df.columns:
        lloom_source_cols.append(optional_col)

lloom_source_df = df[lloom_source_cols].copy()
lloom_source_df = lloom_source_df[lloom_source_df["comment_text_clean"].ne("")].copy()

sample_n = min(LLOOM_DISCOVERY_SAMPLE, len(lloom_source_df))

if (
    "sentiment_label" in lloom_source_df.columns
    and lloom_source_df["sentiment_label"].nunique() > 1
):
    pieces = []
    labels = lloom_source_df["sentiment_label"].dropna().unique().tolist()
    per_label = max(1, sample_n // len(labels))

    for label in labels:
        group = lloom_source_df[lloom_source_df["sentiment_label"] == label]
        pieces.append(group.sample(n=min(per_label, len(group)), random_state=RANDOM_SEED))

    lloom_discovery_df = pd.concat(pieces).drop_duplicates(subset=[ID_COL])

    if len(lloom_discovery_df) < sample_n:
        remaining = lloom_source_df.drop(index=lloom_discovery_df.index, errors="ignore")
        extra_n = min(sample_n - len(lloom_discovery_df), len(remaining))
        if extra_n > 0:
            lloom_discovery_df = pd.concat([
                lloom_discovery_df,
                remaining.sample(extra_n, random_state=RANDOM_SEED),
            ])
else:
    lloom_discovery_df = lloom_source_df.sample(n=sample_n, random_state=RANDOM_SEED)

lloom_discovery_df = lloom_discovery_df.reset_index(drop=True)

print(f"LLooM discovery sample: {len(lloom_discovery_df):,} comments")
if "sentiment_label" in lloom_discovery_df.columns:
    display(lloom_discovery_df["sentiment_label"].value_counts(dropna=False).to_frame("comments"))

lloom_discovery_df.to_csv(OUTPUT_DIR / "lloom_discovery_sample.csv", index=False)

## 18. Initialize LLooM and estimate cost

In [ ]:
if RUN_LLOOM:
    if not os.environ.get("OPENAI_API_KEY"):
        raise EnvironmentError(
            "OPENAI_API_KEY is not set. Set it in your environment, "
            "then rerun this cell."
        )

    import text_lloom.workbench as wb

    l = wb.lloom(
        df=lloom_discovery_df,
        text_col="comment_text_clean",
        id_col=ID_COL,
    )

    print("LLooM initialized.")
    print("Generation cost estimate:")
    display(l.estimate_gen_cost(verbose=True))
else:
    print("RUN_LLOOM is False.")
    print("Set RUN_LLOOM = True after configuring OPENAI_API_KEY.")

## 19. Run LLooM concept induction

`gen_auto()` generates concepts and scores the discovery sample in one workflow. The package normally asks for confirmation before API calls so you can review cost before proceeding.

In [ ]:
if RUN_LLOOM:
    lloom_sample_scores = await l.gen_auto(
        max_concepts=LLOOM_MAX_CONCEPTS,
        seed=LLOOM_SEED,
    )

    lloom_sample_scores.to_csv(
        OUTPUT_DIR / "lloom_discovery_sample_scores.csv",
        index=False,
    )

    lloom_concepts = l.export_df()
    lloom_concepts.to_csv(
        OUTPUT_DIR / "lloom_concepts_summary.csv",
        index=False,
    )

    display(lloom_concepts)
else:
    print("LLooM skipped.")

## 20. Explore LLooM concepts interactively

In [ ]:
if RUN_LLOOM:
    l.vis()

    # Useful optional slices:
    # l.vis(slice_col="sentiment_label", norm_by="concept")
    # l.vis(slice_col="emotion_label", norm_by="concept")
else:
    print("LLooM skipped.")

## 21. Optional: score the full corpus with the induced LLooM concepts

This can create many API calls. For a large dataset, inspect the discovery concepts first. Only enable `LLOOM_SCORE_FULL_DATASET` if the concepts are useful enough to justify full-corpus scoring.

In [ ]:
if RUN_LLOOM and LLOOM_SCORE_FULL_DATASET:
    print(f"Preparing to score {len(lloom_source_df):,} comments against LLooM concepts.")
    print("Estimated scoring cost:")
    display(l.estimate_score_cost(verbose=True))

    lloom_full_scores = await l.score(
        df=lloom_source_df,
        score_all=True,
        batch_size=LLOOM_SCORE_BATCH_SIZE,
    )

    lloom_full_scores.to_csv(
        OUTPUT_DIR / "lloom_full_corpus_scores.csv",
        index=False,
    )

    lloom_full_concept_summary = l.export_df()
    lloom_full_concept_summary.to_csv(
        OUTPUT_DIR / "lloom_full_corpus_concept_summary.csv",
        index=False,
    )

    display(lloom_full_concept_summary)
else:
    print("Full-corpus LLooM scoring not enabled.")

# Part D — Combined analysis tables

## 22. Sentiment and emotion by NMF theme

In [ ]:
if {"nmf_theme_label", "sentiment_label"}.issubset(df.columns):
    sentiment_by_nmf = pd.crosstab(
        df["nmf_theme_label"],
        df["sentiment_label"],
        normalize="index",
    )
    display(sentiment_by_nmf)
    sentiment_by_nmf.to_csv(OUTPUT_DIR / "sentiment_by_nmf_theme.csv")

if {"nmf_theme_label", "emotion_label"}.issubset(df.columns):
    emotion_by_nmf = pd.crosstab(
        df["nmf_theme_label"],
        df["emotion_label"],
        normalize="index",
    )
    display(emotion_by_nmf)
    emotion_by_nmf.to_csv(OUTPUT_DIR / "emotion_by_nmf_theme.csv")

## 23. Theme-level sentiment / toxicity statistics

In [ ]:
agg_map = {"comments": ("comment_text_clean", "size")}

if "sentiment_continuous" in df.columns:
    agg_map["mean_sentiment"] = ("sentiment_continuous", "mean")

if "toxicity" in df.columns:
    agg_map["mean_toxicity"] = ("toxicity", "mean")
    agg_map["max_toxicity"] = ("toxicity", "max")

theme_stats = (
    df.dropna(subset=["nmf_theme_label"])
      .groupby("nmf_theme_label")
      .agg(**agg_map)
      .reset_index()
      .sort_values("comments", ascending=False)
)

display(theme_stats)
theme_stats.to_csv(OUTPUT_DIR / "nmf_theme_sentiment_toxicity_summary.csv", index=False)

## 24. Final export

In [ ]:
final_path = OUTPUT_DIR / "comments_full_analysis.csv"
df.to_csv(final_path, index=False)

print("Final comment-level file:", final_path)
print("\nKey outputs:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print(" -", path)

## Notes for interpretation

- Sentiment, emotion, toxicity, NMF, BERTopic, and LLooM are **different measurements**. Agreement between them is informative, but disagreement is not automatically an error.
- Transformer scores are model outputs, not human judgments. If this analysis supports a paper, validate a sample manually and report the models and preprocessing choices.
- Duplicate/template comments are kept by default because removing them changes observed prevalence.
- LLooM is best treated as a concept-discovery and coding aid. Review concept definitions and representative examples before treating concept prevalence as a substantive finding.
- For a very large corpus, induce LLooM concepts on a sample first, then score the full dataset only after reviewing those concepts.